## Example 1: CloudFormation Template Validation

In [ ]:
import cfnlint
import json

print("Example 1: CloudFormation Template Validation")

try:
    # Show cfn-lint basic info
    print(f"✓ cfn-lint module imported successfully")
    print(f"✓ cfn-lint module: {cfnlint}")
    
    # Sample CloudFormation template
    cf_template = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Description": "Simple S3 bucket template for testing",
        "Resources": {
            "MyTestBucket": {
                "Type": "AWS::S3::Bucket",
                "Properties": {
                    "BucketName": "my-test-bucket-12345",
                    "VersioningConfiguration": {
                        "Status": "Enabled"
                    },
                    "PublicAccessBlockConfiguration": {
                        "BlockPublicAcls": True,
                        "BlockPublicPolicy": True,
                        "IgnorePublicAcls": True,
                        "RestrictPublicBuckets": True
                    }
                }
            }
        },
        "Outputs": {
            "BucketName": {
                "Description": "Name of the S3 bucket",
                "Value": {"Ref": "MyTestBucket"},
                "Export": {
                    "Name": {"Fn::Sub": "${AWS::StackName}-BucketName"}
                }
            }
        }
    }
    
    print("✓ Sample CloudFormation template created")
    print(f"  Template version: {cf_template['AWSTemplateFormatVersion']}")
    print(f"  Resources: {len(cf_template['Resources'])}")
    print(f"  Outputs: {len(cf_template['Outputs'])}")
    
    # Validate template structure
    required_sections = ["AWSTemplateFormatVersion", "Resources"]
    template_valid = all(section in cf_template for section in required_sections)
    print(f"✓ Template structure valid: {template_valid}")
    
    # Check resource types
    resource_types = [resource["Type"] for resource in cf_template["Resources"].values()]
    print(f"✓ Resource types found: {resource_types}")
    
    # Validate S3 bucket configuration
    bucket_resource = cf_template["Resources"]["MyTestBucket"]
    has_versioning = "VersioningConfiguration" in bucket_resource["Properties"]
    has_public_access_block = "PublicAccessBlockConfiguration" in bucket_resource["Properties"]
    
    print("✓ S3 bucket security configuration:")
    print(f"  Versioning enabled: {has_versioning}")
    print(f"  Public access blocked: {has_public_access_block}")
    
    # Show cfn-lint validation capabilities
    print("✓ cfn-lint validation features:")
    print("  - AWS resource property validation")
    print("  - Template syntax and structure checks")
    print("  - Security best practices enforcement")
    print("  - Resource relationship validation")
    print("  - Custom rule implementation support")
    
    print("✓ CloudFormation template validation completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Note: Install with 'pip install cfn-lint'")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfn_lint example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Rule-based Template Analysis

In [ ]:
import cfnlint
from cfnlint.decode.cfn_yaml import loads
import tempfile
import yaml

print("Example 2: Rule-based Template Analysis")

try:
    # Create a more complex CloudFormation template with potential issues
    cf_template_complex = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Description": "EC2 instance with security group",
        "Parameters": {
            "InstanceType": {
                "Type": "String",
                "Default": "t3.micro",
                "AllowedValues": ["t3.micro", "t3.small", "t3.medium"],
                "Description": "EC2 instance type"
            },
            "KeyName": {
                "Type": "AWS::EC2::KeyPair::KeyName",
                "Description": "Name of EC2 KeyPair for SSH access"
            }
        },
        "Resources": {
            "WebServerSecurityGroup": {
                "Type": "AWS::EC2::SecurityGroup",
                "Properties": {
                    "GroupDescription": "Enable HTTP and HTTPS access",
                    "SecurityGroupIngress": [
                        {
                            "IpProtocol": "tcp",
                            "FromPort": 80,
                            "ToPort": 80,
                            "CidrIp": "0.0.0.0/0"
                        },
                        {
                            "IpProtocol": "tcp", 
                            "FromPort": 443,
                            "ToPort": 443,
                            "CidrIp": "0.0.0.0/0"
                        }
                    ]
                }
            },
            "WebServerInstance": {
                "Type": "AWS::EC2::Instance",
                "Properties": {
                    "ImageId": "ami-0abcdef1234567890",
                    "InstanceType": {"Ref": "InstanceType"},
                    "KeyName": {"Ref": "KeyName"},
                    "SecurityGroupIds": [{"Ref": "WebServerSecurityGroup"}],
                    "UserData": {
                        "Fn::Base64": {"Fn::Join": ["", [
                            "#!/bin/bash\n",
                            "yum update -y\n",
                            "yum install -y httpd\n",
                            "systemctl start httpd\n"
                        ]]}
                    }
                }
            }
        },
        "Outputs": {
            "InstanceId": {
                "Description": "EC2 Instance ID",
                "Value": {"Ref": "WebServerInstance"}
            },
            "PublicIP": {
                "Description": "Public IP address",
                "Value": {"Fn::GetAtt": ["WebServerInstance", "PublicIp"]}
            }
        }
    }
    
    print("✓ Complex CloudFormation template created")
    print(f"  Parameters: {len(cf_template_complex['Parameters'])}")
    print(f"  Resources: {len(cf_template_complex['Resources'])}")
    print(f"  Outputs: {len(cf_template_complex['Outputs'])}")
    
    # Show cfn-lint rule categories
    rule_categories = {
        "E": "Errors - Template syntax and structure issues",
        "W": "Warnings - Best practices and potential issues", 
        "I": "Informational - Suggestions for improvement",
        "F": "Fatals - Template parsing failures"
    }
    
    print("✓ cfn-lint rule categories:")
    for category, description in rule_categories.items():
        print(f"  {category}: {description}")
    
    # Show common rule types
    common_rules = [
        "E1001: Basic CloudFormation template format",
        "E2001: Parameter validation",
        "E3001: Resource property validation",
        "W2001: Parameter constraints",
        "W3005: DependsOn referencing resources",
        "W9001: Base64 function usage"
    ]
    
    print("✓ Common validation rules:")
    for rule in common_rules:
        print(f"  - {rule}")
    
    # Analyze template structure
    print("✓ Template analysis:")
    
    # Check parameters
    for param_name, param_config in cf_template_complex["Parameters"].items():
        param_type = param_config.get("Type", "Unknown")
        has_default = "Default" in param_config
        print(f"  Parameter {param_name}: {param_type}, Default: {has_default}")
    
    # Check resource dependencies
    resources = cf_template_complex["Resources"]
    sg_resource = resources["WebServerSecurityGroup"]
    ec2_resource = resources["WebServerInstance"]
    
    # EC2 references security group
    references_sg = "WebServerSecurityGroup" in str(ec2_resource)
    print(f"  EC2 instance references security group: {references_sg}")
    
    # Check security group rules
    ingress_rules = sg_resource["Properties"]["SecurityGroupIngress"]
    open_to_internet = any(rule.get("CidrIp") == "0.0.0.0/0" for rule in ingress_rules)
    print(f"  Security group open to internet: {open_to_internet}")
    
    print("✓ Rule-based template analysis completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfn_lint example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Template Processing and File Handling

In [ ]:
import cfnlint
# `load(filename)` reads a template FILE; `loads(text)` parses template TEXT. The templates below are files.
from cfnlint.decode.cfn_yaml import load as yaml_load
from cfnlint.decode.cfn_json import load as json_load
import tempfile
import json
import yaml

print("Example 3: Template Processing and File Handling")

try:
    # Show cfn-lint file processing capabilities
    print("✓ cfn-lint file processing:")
    
    # Create sample templates in different formats
    lambda_template = {
        "AWSTemplateFormatVersion": "2010-09-09",
        "Description": "AWS Lambda function template",
        "Resources": {
            "LambdaExecutionRole": {
                "Type": "AWS::IAM::Role",
                "Properties": {
                    "AssumeRolePolicyDocument": {
                        "Version": "2012-10-17",
                        "Statement": [{
                            "Effect": "Allow",
                            "Principal": {"Service": "lambda.amazonaws.com"},
                            "Action": "sts:AssumeRole"
                        }]
                    },
                    "Path": "/",
                    "Policies": [{
                        "PolicyName": "LambdaLogs",
                        "PolicyDocument": {
                            "Version": "2012-10-17",
                            "Statement": [{
                                "Effect": "Allow",
                                "Action": [
                                    "logs:CreateLogGroup",
                                    "logs:CreateLogStream",
                                    "logs:PutLogEvents"
                                ],
                                "Resource": "arn:aws:logs:*:*:*"
                            }]
                        }
                    }]
                }
            },
            "HelloWorldFunction": {
                "Type": "AWS::Lambda::Function",
                "Properties": {
                    "Code": {
                        "ZipFile": "def lambda_handler(event, context): return 'Hello World'"
                    },
                    "Handler": "index.lambda_handler",
                    "Runtime": "python3.9",
                    "Role": {"Fn::GetAtt": ["LambdaExecutionRole", "Arn"]},
                    "Timeout": 60,
                    "MemorySize": 128
                }
            }
        },
        "Outputs": {
            "LambdaFunctionArn": {
                "Description": "Lambda Function ARN",
                "Value": {"Fn::GetAtt": ["HelloWorldFunction", "Arn"]}
            }
        }
    }
    
    print("✓ Lambda function template created")
    print(f"  Resources: {len(lambda_template['Resources'])}")
    
    # Test file format handling
    print("✓ Supported file formats:")
    file_formats = {
        "JSON": ".json - Standard JSON CloudFormation format",
        "YAML": ".yaml/.yml - YAML CloudFormation format", 
        "Template": ".template - Generic template format"
    }
    
    for format_name, description in file_formats.items():
        print(f"  {format_name}: {description}")
    
    # Create temporary files for testing
    with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as json_file:
        json.dump(lambda_template, json_file, indent=2)
        json_file_path = json_file.name
        print(f"✓ JSON template written to: {json_file_path.split('/')[-1]}")
    
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as yaml_file:
        yaml.dump(lambda_template, yaml_file, default_flow_style=False)
        yaml_file_path = yaml_file.name
        print(f"✓ YAML template written to: {yaml_file_path.split('/')[-1]}")
    
    # Test template loading capabilities
    print("✓ Template loading capabilities:")
    
    try:
        # Load JSON template
        json_template_loaded = json_load(json_file_path)
        print(f"  JSON template loaded: {len(json_template_loaded.get('Resources', {}))} resources")
        
        # Load YAML template  
        yaml_template_loaded = yaml_load(yaml_file_path)
        print(f"  YAML template loaded: {len(yaml_template_loaded.get('Resources', {}))} resources")
        # Both decoders must give back exactly the template that was written.
        assert json_template_loaded == lambda_template, json_template_loaded
        assert yaml_template_loaded == lambda_template, yaml_template_loaded
        
    except Exception as load_error:
        print(f"  Template loading test: {load_error}")
        raise
    
    # Show validation process
    print("✓ cfn-lint validation process:")
    validation_steps = [
        "1. Template file discovery and loading",
        "2. Syntax and format validation",
        "3. Resource property validation",
        "4. Cross-reference validation",
        "5. Security and best practice checks",
        "6. Custom rule application"
    ]
    
    for step in validation_steps:
        print(f"  {step}")
    
    # Show configuration options
    print("✓ Configuration options:")
    config_options = [
        "Region-specific validation",
        "Custom rule directories", 
        "Rule inclusion/exclusion",
        "Output format selection",
        "Severity level filtering"
    ]
    
    for option in config_options:
        print(f"  - {option}")
    
    print("✓ Template processing and file handling completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfn_lint example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Custom Rules and Advanced Configuration

In [ ]:
import cfnlint
import tempfile
import json
import yaml

print("Example 4: Custom Rules and Advanced Configuration")

try:
    # Show custom rule development concepts
    print("✓ Custom rule development:")
    
    # Mock custom rule structure
    custom_rule_example = {
        "id": "E9999",
        "shortdesc": "Check S3 bucket encryption",
        "description": "Ensure S3 buckets have encryption enabled",
        "source_url": "https://docs.aws.amazon.com/s3/latest/userguide/bucket-encryption.html",
        "tags": ["s3", "security", "encryption"]
    }
    
    print("✓ Custom rule structure:")
    for key, value in custom_rule_example.items():
        print(f"  {key}: {value}")
    
    # Show rule categories and numbering
    rule_numbering = {
        "E1xxx": "Basic CloudFormation template structure",
        "E2xxx": "Parameters",
        "E3xxx": "Resources", 
        "E4xxx": "Metadata",
        "E6xxx": "Outputs",
        "E7xxx": "Mappings",
        "E8xxx": "Conditions",
        "E9xxx": "Custom/Organization specific rules"
    }
    
    print("✓ Rule numbering conventions:")
    for category, description in rule_numbering.items():
        print(f"  {category}: {description}")
    
    # Show configuration options
    print("✓ Advanced configuration options:")
    
    # Configuration file structure
    config_example = {
        "regions": ["us-east-1", "us-west-2", "eu-west-1"],
        "ignore_checks": ["E1001", "W2001"],
        "include_checks": ["E3*", "W3*"],
        "configure_rules": {
            "E2502": {"strict": False},
            "E3008": {"strict": True}
        },
        "custom_rules": "./custom-rules/",
        "override_spec": "./custom-spec.json"
    }
    
    print("✓ Configuration file options:")
    for option, value in config_example.items():
        print(f"  {option}: {value}")
    
    # Show rule matching and filtering
    print("✓ Rule filtering capabilities:")
    filtering_options = [
        "Include specific rules (--include-checks E3*)",
        "Exclude specific rules (--ignore-checks W2001)",
        "Filter by severity (errors, warnings, info)",
        "Region-specific rule application",
        "Resource type filtering"
    ]
    
    for option in filtering_options:
        print(f"  - {option}")
    
    # Show output formats
    output_formats = {
        "default": "Human-readable text format",
        "json": "JSON structured output", 
        "junit": "JUnit XML format for CI/CD",
        "pretty": "Colored and formatted output",
        "sarif": "SARIF format for security tools"
    }
    
    print("✓ Output formats:")
    for format_name, description in output_formats.items():
        print(f"  {format_name}: {description}")
    
    # Show integration capabilities
    integrations = [
        "CI/CD pipeline integration",
        "IDE extensions and plugins",
        "Git pre-commit hooks",
        "AWS CodeBuild/CodePipeline",
        "GitHub Actions workflows"
    ]
    
    print("✓ Integration capabilities:")
    for integration in integrations:
        print(f"  - {integration}")
    
    # Show rule development best practices
    print("✓ Custom rule development best practices:")
    best_practices = [
        "Follow existing rule ID conventions",
        "Provide clear error messages",
        "Include source documentation URLs",
        "Test rules with various template scenarios",
        "Consider regional differences",
        "Implement appropriate severity levels"
    ]
    
    for practice in best_practices:
        print(f"  - {practice}")
    
    print("✓ Custom rules and advanced configuration completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfn_lint example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Integration with Development Workflows

In [ ]:
import cfnlint
import json
import subprocess
import os

print("Example 5: Integration with Development Workflows")

try:
    # Show cfn-lint CLI usage
    print("✓ cfn-lint CLI integration:")
    
    # Command line usage examples
    cli_examples = {
        "Basic validation": "cfn-lint template.yaml",
        "Specific regions": "cfn-lint --regions us-east-1 us-west-2 template.yaml",
        "Ignore warnings": "cfn-lint --ignore-checks W template.yaml",
        "JSON output": "cfn-lint --format json template.yaml",
        "Custom rules": "cfn-lint --append-rules ./custom-rules template.yaml"
    }
    
    print("✓ Command line usage examples:")
    for description, command in cli_examples.items():
        print(f"  {description}: {command}")
    
    # Show CI/CD integration patterns
    print("✓ CI/CD integration patterns:")
    
    # GitHub Actions example
    github_action = """
name: CloudFormation Lint
on: [push, pull_request]
jobs:
  cfn-lint:
    runs-on: ubuntu-latest
    steps:
    - uses: actions/checkout@v2
    - uses: actions/setup-python@v2
      with:
        python-version: '3.9'
    - run: pip install cfn-lint
    - run: cfn-lint templates/*.yaml
"""
    
    print("  GitHub Actions workflow:")
    for line in github_action.strip().split('\n')[:8]:
        print(f"    {line}")
    print("    ... (workflow continues)")
    
    # Pre-commit hook example
    precommit_config = """
repos:
- repo: https://github.com/aws-cloudformation/cfn-lint
  rev: v0.83.8
  hooks:
  - id: cfn-lint-rc
    files: \\.(json|yml|yaml)$
"""
    
    print("  Pre-commit hook configuration:")
    for line in precommit_config.strip().split('\n'):
        print(f"    {line}")
    
    # Show IDE integration
    ide_integrations = [
        "VS Code: AWS Toolkit extension with cfn-lint",
        "IntelliJ: CloudFormation plugin integration",
        "Vim: ALE (Asynchronous Lint Engine) support",
        "Emacs: Flycheck integration",
        "Sublime Text: SublimeLinter plugin"
    ]
    
    print("✓ IDE integrations:")
    for integration in ide_integrations:
        print(f"  - {integration}")
    
    # Show reporting and metrics
    print("✓ Reporting and metrics:")
    
    # Mock validation results structure
    validation_results = {
        "template": "infrastructure.yaml",
        "errors": 2,
        "warnings": 5,
        "informational": 3,
        "total_issues": 10,
        "rules_applied": 245,
        "execution_time": "1.2s"
    }
    
    print("  Sample validation report:")
    for metric, value in validation_results.items():
        print(f"    {metric}: {value}")
    
    # Show automation capabilities
    automation_scenarios = [
        "Validate templates on git commit",
        "Block deployment of non-compliant templates",
        "Generate compliance reports",
        "Integration with AWS Config rules",
        "Automated remediation suggestions"
    ]
    
    print("✓ Automation scenarios:")
    for scenario in automation_scenarios:
        print(f"  - {scenario}")
    
    # Show best practices for workflow integration
    print("✓ Workflow integration best practices:")
    best_practices = [
        "Run validation early in development cycle",
        "Configure appropriate rule exclusions",
        "Use region-specific validation",
        "Implement graduated severity handling",
        "Maintain consistent rule configuration",
        "Monitor and track template quality metrics"
    ]
    
    for practice in best_practices:
        print(f"  - {practice}")
    
    # Show ecosystem integration
    ecosystem_tools = {
        "AWS SAM": "Integrated CloudFormation linting",
        "AWS CDK": "Template synthesis validation",
        "Terraform": "CloudFormation output validation",
        "Pulumi": "CloudFormation export validation",
        "Troposphere": "Generated template validation"
    }
    
    print("✓ Ecosystem tool integration:")
    for tool, description in ecosystem_tools.items():
        print(f"  {tool}: {description}")
    
    print("✓ Integration with development workflows completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"cfn_lint example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")